In [ ]:
# # Environment Testing :
# from langchain_openai import ChatOpenAI
# import os 
# from dotenv import load_dotenv

# load_dotenv()

# GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")

# llm = ChatOpenAI(
#     model="gpt-4.1-mini",
#     base_url="https://models.github.ai/inference",
#     temperature=0.1,
#     max_tokens=500,
#     api_key=GITHUB_TOKEN
# )

# response = llm.invoke("What is the capital of India?")
# print(response.content)

### Read PDF Files:

In [ ]:
# Load Libraries:
from langchain_community.document_loaders import PyMuPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path
import os 

# Read All Pdf's :
def process_pdf(pdf_direcory):
    
    all_documents = []
    pdf_dir = Path(pdf_direcory)

    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} pdf process")
    
    for p_file in pdf_files:
        print(f"\nProcess : {p_file}")

        try:
            loader = PyMuPDFLoader(str(p_file))
            documents = loader.load()

            # Add Source Info for metadata:
            for doc in documents:
                doc.metadata['source_file'] = p_file.name
                doc.metadata['file_type'] = "pdf"

            all_documents.extend(documents)
            print(f"💯Loded {len(documents)} Pages")
        
        except Exception as e:
            print(f"✖️ Error {e}")
        
    print(f"Total Documents Loded: {len(all_documents)}")
    return all_documents



all_pdf_documents = process_pdf(r"D:\Code\AI-ML\Conversational_RAG\Data")

### Chunking Documnets:

In [ ]:
# Text Splitting get into chunks
def split_documnets(documnets, chunk_size=1000, overlap = 200):
    """Split Documents into Chunks."""
    text_chunk = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap = overlap,
        length_function = len,
        separators= ["\n\n","\n"," ", ""] #\n\n is a Peragraph Seprator
    )

    split_doc = text_chunk.split_documents(documnets)
    print(f"Split {len(documnets)} Documnets into {len(split_doc)} chunks.")

    if split_doc:
        print(f"\nExample Chunk:\nContent : \n{split_doc[0].page_content[:200]}...\nMetaData: {split_doc[0].metadata}")
    
    return split_doc

chunks = split_documnets(all_pdf_documents)


### vectore Store:

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import SentenceTransformerEmbeddings
import os

def vector_store(chunks, save_path="faiss_index"):
    
    print("\n🔄 Creating Embeddings (FAISS)...")

    embedding_model = SentenceTransformerEmbeddings(
        model_name="all-MiniLM-L6-v2"
    )

    vector_db = FAISS.from_documents(
        documents=chunks,
        embedding=embedding_model
    )

    # Ensure directory exists
    os.makedirs(save_path, exist_ok=True)

    # Save locally
    vector_db.save_local(save_path)

    print(f"✅ FAISS Vector DB Created with {len(chunks)} chunks")
    print(f"📁 Saved at: {save_path}")

    return vector_db

vectore_db = vector_store(chunks,"D:\Code\AI-ML\Conversational_RAG\Vectore_DB")

### Chat with LLM:

In [ ]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_openai import ChatOpenAI
import os

def start_chat(vector_db):
    """Start conversational RAG chatbot."""

    print("\n🤖 Chatbot Started (type 'exit' to stop)\n")

    # LLM
    llm = ChatOpenAI(
        model="gpt-4.1-mini",
        base_url="https://models.github.ai/inference",
        temperature=0.1,
        max_tokens=500,
        api_key=os.getenv("GITHUB_TOKEN")
    )

    # Memory
    memory = ChatMessageHistory()

    # Retriever
    retriever = vector_db.as_retriever(search_kwargs={"k": 3})

    while True:
        query = input("User: ")

        if query.lower() == "exit":
            print("\n👋 Exiting chatbot...")
            break

        # 🔍 Retrieve docs
        docs = retriever.invoke(query)

        if not docs:
            answer = "I don't know based on the provided documents."
        else:
            # 📄 Context
            context = "\n\n".join([doc.page_content for doc in docs])

            # 💬 Chat history
            chat_history = "\n".join([
                f"User: {msg.content}" if msg.type == "human" else f"Bot: {msg.content}"
                for msg in memory.messages
            ])

            # 🧠 Strict prompt
            final_prompt = f"""
            You must answer ONLY using the context below.

            If the answer is NOT explicitly present, reply exactly:
            "I don't know based on the provided documents."

            ---------------------
            CONTEXT:
            {context}
            ---------------------

            CHAT HISTORY:
            {chat_history}

            QUESTION:
            {query}

            ANSWER:
            """

            response = llm.invoke(final_prompt)
            answer = response.content.strip()

        # 💾 Save memory
        memory.add_user_message(query)
        memory.add_ai_message(answer)

        # 🖨️ PRINT FULL CONVERSATION
        print("\n------ Conversation ------")
        for msg in memory.messages:
            if msg.type == "human":
                print(f"👤User: {msg.content}")
            else:
                print(f"🤖Bot: {msg.content}")
        print("--------------------------\n")

In [ ]:
chat = start_chat(vectore_db)